# Seeded reasoning: do ideas from distant fields help Qwen solve math?

**Kaggle settings:** Accelerator GPU T4 x2, Internet ON.
Plan: pilot (30 problems, about 720 generations) to measure speed and check outputs, then the main run
(200 problems x 6 conditions x 8 samples). Runs are resumable: if the session stops, re-run the same cell.

In [ ]:
!git clone -q https://github.com/Mathindata/seeded-reasoning-lab.git
%cd seeded-reasoning-lab
!pip install -q math-verify
import os, sys, json
sys.path.insert(0, "src"); os.environ["PYTHONPATH"] = "src"

In [ ]:
# vLLM is much faster. If this install or the model start fails on T4, set BACKEND = "hf" below.
!pip install -q vllm
BACKEND = "vllm"
# If vLLM complains about the GPU, try the older engine first:  os.environ["VLLM_USE_V1"] = "0"

In [ ]:
# 1. Pilot: speed and sanity check (watch the gen/s line to estimate the main run)
!python -m sr run --config configs/pilot.json --backend $BACKEND

In [ ]:
# 2. Read a few outputs before trusting any number
import pandas as pd
df = pd.read_json("runs/pilot_v2/results.jsonl", lines=True)
print("truncated share:", df.truncated.mean().round(3), "| no boxed answer:", (df.pred == "").mean().round(3))
print(df.groupby("condition")[["truncated", "correct"]].mean().round(3))
for _, r in df[df.condition == "far_translated"].head(5).iterrows():
    print("\nSEED:", r.domain, "|", r.seed, "\nSTRATEGY:", r.strategy, "\nCORRECT:", r.correct)

In [ ]:
# 3. Main run (resumable). Estimated time = 16,000 generations (now up to 2,048 tokens each) / gen/s from the pilot.
!python -m sr run --config configs/main.json --backend $BACKEND

In [ ]:
# 4. Analysis
from IPython.display import Image, display
from sr.analyze import analyze
res = analyze("runs/main")
print(res["info"]); display(res["summary"].round(3))
c = res["comparisons"]; display(c.round(3))
display(res["mixture"].round(3))
display(Image("runs/main/analysis/pass_at_k.png")); display(Image("runs/main/analysis/diff_vs_plain.png"))

In [ ]:
# 5. Optional: Game of 24 (search-heavy; exact checking)
# !python -m sr run --config configs/game24.json --backend $BACKEND

In [ ]:
# 6. Save everything for download
!zip -qr runs.zip runs
print("download runs.zip from the Output panel")